# Fit Models

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.base import clone

from tqdm_joblib import tqdm_joblib
from tqdm.auto import tqdm

from bioacoustics.features import augment_temporal_features, augment_sites

from bioacoustics.data import load_results, save_results
from bioacoustics.metrics import evaluate_multilabel_model

from bioacoustics.modeling import FitMode, Classifier, HierarchicalMixtureOfExperts
from bioacoustics.modeling import split_data, get_prediction_pipeline
from bioacoustics.modeling import smooth_proba


from bioacoustics.modeling import get_feature_importance

from bioacoustics.modeling import ignore_warnings

import bioacoustics.visualization  as viz 

ignore_warnings()  # TODO: fix the reasons for these warnings

%load_ext autoreload
%autoreload 2

Problems to think about:

- Multi-label prediction -> hierarchical models? implement with `Pipeline`
- Class imbalance -> tinker with class weights, adjust probabilities?
- Use additional metadata from train, not available in soundscapes
- Maybe use hybrid approaches? learn about how to combine different models' predictions

think about chunking the audio, tuning the threshold for ech class

## Load features and labels

In [ ]:
data_train = load_results("features", "data_train")
data_train_soundscapes = load_results("features", "data_train_soundscapes")

# Decided to not keep them as the gain in ROC-AUC is
# marginal (~1%) and they introduce potential recording bias
AUGMENT_WITH_TIME = False
AUGMENT_WITH_SITES = False

if AUGMENT_WITH_TIME:
    data_train_soundscapes["X"] = augment_temporal_features(data_train_soundscapes["X"])
if AUGMENT_WITH_SITES:
    data_train_soundscapes["X"] = augment_sites(data_train_soundscapes["X"])

Note that it doesn't make much sense to validate on iNat or XC data, since it's of different format than test soundscapes.

In [ ]:
FIT_MODE = FitMode.MIX_TO_SOUNDSCAPE

X_train, X_test, y_class_train, y_class_test, y_primary_train, y_primary_test = (
    split_data(
        data_train,
        data_train_soundscapes,
        FIT_MODE,
        test_size=0.2,
        random_state=41,
        rare_first=False
    )
)

About metrics:

The metric they use in BirdClef is macro ROC-AUC a a version of
macro-averaged ROC-AUC that skips classes that have no true positive labels.

- Macro - metric computed per class then averaged, giving each label
equal weight regardless of frequency.

- Micro - metric computed globally by aggregating all true positives,
false positives, and false negatives across labels.

- Hamming loss - fraction of incorrectly predicted label assignments
across all samples and labels.

- LRAP (Label Ranking Average Precision) - evaluates how well true
labels are ranked above others for each sample; averages the rank
quality over true labels and samples.

- Macro metrics are computed only over classes that have at least one
positive sample in the test set; zero-support classes would make
roc_auc_score return NaN and contribute uninformative zeros to F1.

In [ ]:
SMOOTH_SIGMA = 4 # 0 if no smoothing

## Baseline species prediction

This is the model we are using for hyperparameter tuning and model selection with cross validation on train data only. The validation set defined above is used for final validation and test on more resource-consuming approaches such as MoE.

The models we will try are:
- XGBoost - most promising candidate
- Random Forest 
- Logistic Regression - as a linear baseline

We decided to exclude SVM because linear kernel is not very useful for this non-linear data, and non-linear kernels will be slow to tune.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, make_scorer

N_CV_FOLDS = 3

def _macro_roc_auc_cv(y_true, y_score):
    """Macro ROC AUC over classes with at least one positive — mirrors the BirdClef metric."""
    y_arr = y_true.values if hasattr(y_true, "values") else y_true
    support = y_arr.sum(axis=0) > 0
    if support.sum() < 2:
        return np.nan
    return roc_auc_score(y_arr[:, support], y_score[:, support], average="macro")

cv_scorer = make_scorer(_macro_roc_auc_cv, response_method="predict_proba")

param_grids = {
    Classifier.XGBOOST: {"clf__estimator__max_depth": [4, 6]},
    Classifier.RF:      {"clf__estimator__max_depth": [10, None]},
    Classifier.LR:      {"clf__estimator__C": [0.1, 1.0, 10.0]}, # TODO: doesn't seem to converge
}

cv_split = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)

gs_results = {}
for clf_type, param_grid in param_grids.items():
    print(f"\n--- {clf_type.name} ---")
    pipeline = get_prediction_pipeline(clf_type)
    pipeline.set_params(clf__n_jobs=1)  # avoid nested parallelism inside CV folds
    gs = GridSearchCV(
        pipeline, param_grid,
        scoring=cv_scorer, cv=cv_split, refit=False, verbose=1, n_jobs=3,
    )
    gs.fit(X_train, y_primary_train)
    gs_results[clf_type] = gs
    print(f"  Best: {gs.best_params_}  CV score: {gs.best_score_:.4f}")

best_classifier = max(gs_results, key=lambda c: gs_results[c].best_score_)
best_params = {k.split("__")[-1]: v for k, v in gs_results[best_classifier].best_params_.items()}
print(f"\n=> Best classifier: {best_classifier.name}, params: {best_params}")

In [ ]:
CLASSIFIER = best_classifier 

pipeline_baseline = get_prediction_pipeline(CLASSIFIER, **best_params)
with tqdm_joblib(desc="Training OvR", total=y_primary_train.shape[1]):
    pipeline_baseline.fit(X_train, y_primary_train)

In [ ]:
print(f"{FIT_MODE.name} predicting SPECIES".center(60))
print("(ignore CLASS)".center(60))
eval_report =  evaluate_multilabel_model(pipeline_baseline, X_test, y_primary_test, smooth_sigma=SMOOTH_SIGMA)
save_results(eval_report, f"{CLASSIFIER.name.lower()}-{FIT_MODE.name.lower()}-species_baseline")

## Taxonomy to predict species

We can use these approaches:

1. **Hierarchical classification:**
    - first predict the class and then train per-class model
    - we must avoid training mismatch - at inference predicted class is used, but at training use only true class, so we better condition on predicted class
    - problems:
        - error propagation if predicted class is wrong
    - **soft version:**
        - $P(s|x) = \sum_c P(s|x,c)P(c|x)$
        - predict class probabilities and use them as soft weights for species classifiers
        - i.e. mixture of experts, soft routing
        - pros:
            - avoids routing errors
            - more consistent as a probability model
    - ideas:
        - I'm afraid class signal will be hidden by hundreds of features, maybe we should emphasize it?
    - multilabel problem:
        - since data is multilabel, classes are not mutually exclusive, it means that when we do an averages sum of preicted probabilities, it will not sum to 1
        - so we would need to normalize it or just do average prediction across experts

1. **Feature augmentation**
    - include predicted class as a new feature:
    - no hard routing, so more robust
    - simple to implement
    - problems:
        - this feature can be noisy
    - **soft version:**
        - include predicted class probabilities as new features

1. Other ideas:
    - hierarchical loss (penalize wrong species but good class less)

We would focus on soft approaches as they are more robust since they avoid noisy hard class assignment.

### Predict class



In [ ]:
pipeline_class = get_prediction_pipeline(CLASSIFIER)
with tqdm_joblib(desc="Training OvR", total=y_class_train.shape[1]):
    pipeline_class.fit(X_train, y_class_train)

In [ ]:
print(f"{FIT_MODE.name} predicting CLASS".center(60))
eval_report = evaluate_multilabel_model(pipeline_class, X_test, y_class_test)
save_results(eval_report, f"{CLASSIFIER.name.lower()}-{FIT_MODE.name.lower()}-class")

In [ ]:
N_FOLDS = 5

# use out-of-fold (OOF) prediction unbiased class probabilities for training samples
y_class_train_proba = cross_val_predict(
    clone(pipeline_class), X_train, y_class_train,
    cv=KFold(n_splits=N_FOLDS, shuffle=True, random_state=42),
    method="predict_proba",
)
y_class_test_proba = pipeline_class.predict_proba(X_test)

### Soft hierarchical classification

In [ ]:
expert_pipelines = HierarchicalMixtureOfExperts(y_class_test.shape[1], CLASSIFIER)
expert_pipelines.fit(X_train, y_class_train, y_primary_train)

In [ ]:
print(f"{FIT_MODE.name} predicting SPECIES".center(60))
print("(soft mixture of CLASS experts)".center(60))
eval_report = evaluate_multilabel_model(expert_pipelines, X_test, y_primary_test, y_class_test_proba, smooth_sigma=SMOOTH_SIGMA)
save_results(eval_report, f"{CLASSIFIER.name.lower()}-{FIT_MODE.name.lower()}-species_moe")

### Soft feature augmentation

In [ ]:
X_test_augmented = np.concatenate([X_test, y_class_test_proba], axis=1)
X_train_augmented = np.concatenate([X_train, y_class_train_proba], axis=1)

pipeline_augmentation = get_prediction_pipeline(CLASSIFIER)

with tqdm_joblib(desc="Training OvR", total=y_primary_train.shape[1]):
    pipeline_augmentation.fit(X_train_augmented, y_primary_train)

In [ ]:
print(f"{FIT_MODE.name} predicting SPECIES".center(60))
print("(CLASS feature augmentation)".center(60))
eval_report = evaluate_multilabel_model(pipeline_augmentation, X_test_augmented, y_primary_test)
save_results(eval_report, f"{CLASSIFIER.name.lower()}-{FIT_MODE.name.lower()}-species_feat_augmentation")

## Interpret the model

In [ ]:
for pipeline in (pipeline_class, pipeline_baseline):
    class_names = y_class_train.columns
    primary_names = y_primary_train.columns
    df_importance = get_feature_importance(pipeline, class_names)
    TOP_N = 20
    viz.plot_importance_heatmap(df_importance, top_n=TOP_N)
    viz.plot_importance_mean(df_importance, top_n=TOP_N)

## Closer inspection of errors

In [ ]:
y_proba = pipeline_class.predict_proba(X_test)
viz.plot_multilabel_errors(y_class_test, y_proba, threshold=0.5)

In [ ]:
y_proba = pipeline_baseline.predict_proba(X_test)
viz.plot_multilabel_errors_large(y_primary_test, y_proba, threshold=0.5)

# SANDBOX

In [ ]:
# TODO: the same idea for training metadata that is absent for the test
# - we can learn to predict this metadata as a secondary task and then include
# it in the model

# NOTE: attention to data leak between train - validation when using secondary tasks
# to generate features (class, metadata)

# TODO: can use this metadata as well for stratification - better split validation
# (make sure that there is no data leak because of the same site)
# TODO: check whether metadata alone can predict species (check for shortcut learning)

# TODO: smart cross validations

# TODO: quality of train audio from iNat and xeno-cant is poorer and further from test than train soundscapes,
#       so maybe we should use them only for species poorly covered by soundscapes

# TODO: impute some NaNs

# TODO: maybe stratify be species or sth else since train data had
#   too (really) many bird recordings, whereas soundscapes contain more amphibians

# TODO: maybe play with thresholds

# TODO: temporal smoothing of predicted probabilities!